
#12 — External Fragrance Sources Evaluation

## Objective

Evaluate Scento's perfume database, Scento's live API, and the 10ML Perfume API as potential reference sources for the Fragrance Recommendation Platform.

## Scope

1. Inspect source schemas and data quality.
2. Explore perfumes, houses, materials, notes and accords.
3. Test API coverage using representative perfume queries.
4. Compare results with existing project data in a later stage.

## Data governance

- Preserve existing source and cleaned tables.
- Keep external sources in separate staging tables.
- Attribute Scento under CC BY 4.0.
- Cache API responses and avoid unnecessary requests.


In [0]:
import requests
import pandas as pd
from io import StringIO
from pyspark.sql import functions as F

SCENTO_BASE = (
    "https://huggingface.co/datasets/"
    "scento/perfume-database/resolve/main"
)

SCENTO_API = "https://www.scento.com/api/perfume/v1"
TENML_API = "https://mcp.10ml.com/api/v1"

HEADERS = {
    "User-Agent": "FragranceRecommendationPlatform/0.1"
}

SOURCE_FILES = {
    "perfumes": f"{SCENTO_BASE}/perfumes.csv",
    "houses": f"{SCENTO_BASE}/houses.csv",
    "materials": f"{SCENTO_BASE}/materials.csv"
}

In [0]:
scento_data = {}

for name, url in SOURCE_FILES.items():
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text))
    scento_data[name] = df

    print(f"\n{name.upper()}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(df.dtypes)
    display(df.head(5))

In [0]:
def scento_get(endpoint, params=None):
    response = requests.get(
        f"{SCENTO_API}/{endpoint.lstrip('/')}",
        params=params,
        headers=HEADERS,
        timeout=20
    )
    response.raise_for_status()
    return response.json()


# Test a known perfume
layton = scento_get("perfumes/layton")

display(layton)

In [0]:

target_tables = {
    "perfumes": "main.fragrance_analytics.scento_perfumes",
    "houses": "main.fragrance_analytics.scento_houses",
    "materials": "main.fragrance_analytics.scento_materials",
}

for name, table_name in target_tables.items():
    pandas_df = scento_data[name]
    spark_df = spark.createDataFrame(pandas_df)

    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"Saved {spark_df.count():,} rows to {table_name}")

In [0]:
# Install only if the package is not already available
%pip install perfume-api

In [0]:
from perfume_api import (
    search_perfumes,
    get_perfume,
    similar_perfumes,
    perfume_alternatives,
    compare_perfumes,
    explain_note
)

search_result = search_perfumes(
    "Layton Parfums de Marly",
    limit=5
)

display(search_result)

In [0]:

layton_profile = get_perfume("parfums-de-marly-layton-edp")

print(layton_profile)


In [0]:

layton_id = "parfums-de-marly-layton-edp"

similar_result = similar_perfumes(layton_id)
alternatives_result = perfume_alternatives(layton_id)

print("SIMILAR PERFUMES")
print(similar_result)

print("\nALTERNATIVES")
print(alternatives_result)

In [0]:
comparison_result = compare_perfumes([
    "parfums-de-marly-layton-edp",
    "al-haramain-detour-noir-edp"
])

print(comparison_result)

# Fragrance Recommendation Platform — Data Source Inventory

**Last updated:** 29 October 2026  
**Platform:** Databricks  
**Purpose:** Maintain an inventory of fragrance, product, price and recommendation data sources for the Fragrance Recommendation Platform.

## 1. Current data landscape

The project has three main data categories:

1. **Marketplace data** — products, listings, brands, conditions and prices.
2. **Fragrance reference data** — perfume identities, notes, accords, fragrance families, ratings and perfumers.
3. **Recommendation enrichment data** — similarity relationships, alternative fragrances, materials and descriptive attributes.

The sources will initially be stored and processed independently. Integration will happen after source coverage, data quality, matching confidence and usage rights have been assessed.

## 2. Dataset inventory

### A. Databricks — Luxury Perfume Prices - FULL

**Status:** Loaded into Databricks; primary marketplace source.

**Purpose:** Analyze available fragrance listings and prices, identify products, and ultimately connect recommendations to purchasable listings.

**Known fields include:**
- Brand and listing title
- Listing type and condition
- Price and price-with-currency
- Seller and shipping information
- Item location and last-updated timestamp
- Perfume type

**Existing table:** `main.fragrance_analytics.marketplace_base`

**Planned processing:**
- Preserve the original source as an immutable raw table.
- Exclude wholesale listings and genuinely unidentifiable products.
- Retain identifiable special formats, including sets, bundles, testers, empty bottles and display bottles.
- Standardize brand and product names.
- Classify product format and clean price information.
- Match identifiable products to fragrance reference sources.

**Primary role:** Marketplace inventory and price analysis.

### B. Fragrantica — Kaggle dataset

**Source:** [Fragrantica.com Fragrance Dataset — Kaggle](https://www.kaggle.com/datasets/olgagmiufana1/fragrantica-com-fragrance-dataset/data)

**Status:** Existing reference data is available in Databricks; the exact relationship between that snapshot and the current Kaggle download should be verified.

**Known existing reference location:** `/Volumes/main/fragrance_analytics/reference_data`

**Known reference data fields:**
- Name
- Gender
- Rating
- Main Accords
- Perfumers
- Description
- URL

The existing reference contains approximately 24,063 records.

**Planned use:**
- Identify fragrances corresponding to cleaned marketplace brands and titles.
- Analyze accords, ratings, popularity and perfumer information.
- Support fragrance preference profiles and recommendation ranking.
- Reuse the existing reference rather than importing a duplicate without checking versions.

**Important considerations:** Record the snapshot date and review the dataset's licence and attribution requirements before redistribution or public deployment.

### C. Scento — Perfume Database and API

**Sources:**
- [Scento Perfume Database](https://www.scento.com/perfume-database)
- [Scento Perfume API documentation](https://www.scento.com/developers/perfume-api)
- [Scento dataset repository](https://github.com/Scento-com/perfume-dataset)

**Status:** Explored; three source tables saved in Databricks.

**Saved tables:**
- `main.fragrance_analytics.scento_perfumes`
- `main.fragrance_analytics.scento_houses`
- `main.fragrance_analytics.scento_materials`

**Data coverage observed in the downloaded snapshot:**

| Dataset | Rows | Main content |
|---|---:|---|
| Perfumes | 1,479 | Names, houses, years, perfumers, concentration, families, note pyramids, seasons and descriptions |
| Houses | 160 | House names, countries, founding years, classifications and descriptions |
| Materials | 420 | Material names, aliases, material families, origin and smell descriptions |

Scento documents its perfume notes as sourced from fragrance houses or authorized retailers, with source information available on individual perfume pages. Its published dataset is licensed under CC BY 4.0; attribution and applicable terms should be retained when using it.

**Planned use:**
- Enrich matched fragrance identities with additional attributes.
- Obtain top, heart and base notes where recorded.
- Explore raw materials and note aliases.
- Add fragrance family, season and time-of-day attributes.
- Cross-check reference data while preserving source provenance.

**Important limitation:** Scento's fragrance families and classifications are its own taxonomy and should not be treated as equivalent to Fragrantica Main Accords.

### D. 10ML — Fragrance Index API

**Source:** 10ML Fragrance Index, accessed through the `perfume-api` Python client.

**Status:** API functions explored; persistent Databricks tables have not yet been established.

**Functions tested:**
- `search_perfumes()`
- `get_perfume()`
- `similar_perfumes()`
- `perfume_alternatives()`
- `compare_perfumes()`

**Observed data fields:**
- Perfume ID, name and house
- Year and gender classification, where available
- Fragrance family
- Accords and key notes
- Descriptive character
- Similar-fragrance results
- Curated alternatives

The comparison function retrieves perfume profiles and compares their notes and accords locally. Similarity results and editor-curated alternatives are distinct outputs and should be stored separately if collected.

**Planned use:**
- Generate additional candidate fragrances for similarity-based recommendations.
- Enrich matched fragrance profiles.
- Explain recommendations using shared notes and accords.
- Explore editor-curated alternatives as a separate recommendation signal.

**Important limitations:**
- Profiles may have different levels of completeness.
- Missing attributes must not be interpreted as confirmed absence.
- 10ML's taxonomy should remain separate from other sources.
- Check the API's terms and permitted persistence before collecting data at scale.

### E. Kaggle — Perfume Response Analysis

**Source:** [Perfume Response Analysis — Kaggle](https://www.kaggle.com/datasets/minahilfatima12328/perfume-response-analysis/data)

**Status:** Identified; schema and data quality not yet assessed.

**Potential use:** Depends on the actual fields. Evaluate whether it contains consumer responses, ratings, preferences or other observations that could help distinguish descriptive similarity from user preference.

**Next step:** Inspect the files, columns, sample rows, record count, missing values and licence.

### F. Kaggle — Perfume Dataset

**Source:** [Perfume Dataset — Kaggle](https://www.kaggle.com/datasets/ayushghawana/perfume-dataset)

**Status:** Identified; schema and overlap not yet assessed.

**Potential use:** Potential supplementary attributes such as perfume type, category, target audience or longevity, depending on the downloaded version.

**Next step:** Inspect the schema and compare brand–product coverage against existing sources before deciding whether to retain it.

### G. Kaggle — Perfume Recommendation Dataset

**Source:** [Perfume Recommendation Dataset — Kaggle](https://www.kaggle.com/datasets/nandini1999/perfume-recommendation-dataset)

**Status:** Identified; schema and data quality not yet assessed.

**Potential use:** Explore perfume descriptions and notes for text-based similarity, feature extraction and recommendation experiments.

**Next step:** Assess whether its product coverage, note fields and descriptive text add information beyond the existing Fragrantica reference.

### Initial status summary

| Source | Loaded in Databricks | Main role |
|---|---|---|
| Luxury Perfume Prices - FULL | Yes | Marketplace and prices |
| Fragrantica reference | Yes | Ratings, accords and fragrance information |
| Scento | Yes | Notes, materials and houses |
| 10ML | API explored | Similarity and curated alternatives |
| Perfume Response Analysis | Not yet assessed | Potential consumer-response data |
| Ayush Ghawana Perfume Dataset | Not yet assessed | Potential supplementary attributes |
| Perfume Recommendation Dataset | Not yet assessed | Potential recommendation features |

## 3. Additional source to evaluate

### FragDB — Fragrance Database

**Source:** [FragDB on GitHub](https://github.com/FragDB/fragrance-database)

FragDB advertises a structured fragrance catalogue, brand and perfumer information, notes, accords, translations and companion datasets containing user reviews and editorial content. Its published repository describes a much larger catalogue than the existing Fragrantica reference, together with multilingual text data.

**Why it may be valuable:**
- Potentially broader fragrance and brand coverage.
- Structured relationships between fragrances, notes, accords, brands and perfumers.
- Consumer review text that could eventually support preference modelling or sentiment analysis.
- Multilingual data that may help with international product matching.

**Caution:** Much of this source's value may overlap with Fragrantica-derived data. Before adopting it, inspect provenance, licence, redistribution restrictions, source coverage and overlap with the existing snapshot. Treat its advertised scale as something to verify against the actual files.

**Status:** Candidate for evaluation, not yet adopted.

## 4. Data governance principles

1. Keep original source data separate and reproducible.
2. Record source, snapshot date, licence and retrieval date.
3. Preserve each source's original taxonomy and raw attributes.
4. Maintain explicit mappings between source-specific fragrance identities.
5. Do not treat fuzzy name matches as confirmed product identities.
6. Distinguish missing information from confirmed absence.
7. Separate similarity results, editor-curated alternatives and recommendations generated by our own model.
8. Check licensing and redistribution rights before publishing datasets or deploying a public application.

## 5. Recommended next steps

1. Inspect the three remaining Kaggle datasets and record their actual schemas.
2. Verify the exact Fragrantica snapshot already stored in Databricks.
3. Evaluate FragDB's provenance, licensing and incremental value.
4. Compare source coverage and overlap using cleaned brand and product identities.
5. Select the sources that add genuinely complementary information.
6. Build the curated cross-source reference layer only after these checks.

**Guiding principle:** Collect broadly enough to understand the available data, but integrate selectively. The objective is not to accumulate the largest number of datasets; it is to build a reliable fragrance reference and recommendation system.